# 44 · Analysis: what graduates actually do, and the ACS test of the major → job bridge

**Plan:** (nb 44) and §3e (the test rule, written before this ran). Reads: `fct_grad_outcome` (ACS PUMS 2020-2024, NYC residents, civilian employed, not enrolled; four cuts, four age bands, four degree levels; D17), `bridge_major_job` and `ref.ref_fod_cip` (to rebuild the linked job set per field), `dim_job` (entry education, major groups, degree-reachable flag), `fct_job_projection` 2022-32 (openings, for context). Changes nothing.

**Questions**
1. Where do NYC bachelor's holders aged 22–34 actually work, and how many are in jobs that do not ask for a bachelor's?
2. What does each field's graduate do: which job families, how spread out?
3. **The D16 test:** are Psychology, Political Science and Criminal Justice graduates found in the jobs the bridge sends their supply to (Managers All Other; Social Science Research Assistants; police supervisors), more than any bachelor's holder is?
4. What do fields earn, does holding a linked job pay, and who goes on to a graduate degree?
5. **Added (plan §3h):** does the graduate degree pay off, by field (§5c)?

**Rules (§3e):** headline = bachelor's-only holders aged 22–34; the bridge under test = the `field_x_linked` set (bachelor's rows of `bridge_major_job`, All Other in); lift = linked share ÷ all-bachelor's base rate, cutoffs 3 / 1.5; D16 read at field × SOC major group, 1.5× the all-bachelor's share.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from explore import q

pd.set_option("display.max_columns", 40, "display.max_colwidth", 60, "display.width", 240,
              "display.float_format", "{:,.1f}".format)
TEST = {"5200": "Psychology", "5506": "Political Science And Government", "5301": "Criminal Justice And Fire Protection"}
CONTRAST = ["6107", "2102", "6201", "6207", "5501", "3600", "3301", "6203"]   # nursing, CS, accounting, finance, economics, biology, english, business admin


def md(s):
    display(Markdown(s))


def show(df, title=None):
    if title:
        md(f"**{title}**")
    display(df)


G = q("""
select cut, group_1, group_1_label, group_2, group_2_label, age_band, degree_level, n_unweighted, n_ftyr, suppressed, earnings_suppressed,
       workers, moe_workers, share_of_group, moe_share, median_earnings_2024usd, moe_median_earnings, mean_earnings_2024usd, group_workers,
       includes_allocated_codes
from fct_grad_outcome
""")
jobs = q("select soc_code, soc_title, soc_level, major_code, entry_education, entry_education_rank, is_degree_entry, is_all_other from dim_job")
print(f"fct_grad_outcome rows: {len(G):,} (yml: 21,791); unsuppressed {int((~G.suppressed).sum()):,} (3,260); with earnings {int((~G.earnings_suppressed).sum()):,} (2,618)")
show(G.groupby("cut").agg(rows=("cut", "size"), unsuppressed=("suppressed", lambda s: int((~s).sum())), fields=("group_1", "nunique"), jobs=("group_2", "nunique")), "Rows per cut")

fct_grad_outcome rows: 21,791 (yml: 21,791); unsuppressed 3,260 (3,260); with earnings 2,618 (2,618)


**Rows per cut**

,rows,unsuppressed,fields,jobs
cut,,,,
degree_x_soc,7144,1378,1,636
field_total,1269,538,174,1
field_x_linked,1676,681,78,3
field_x_soc_major,11702,663,174,22


## 1. Who is in the data
`group_workers` is the share denominator and is identical on every row of a group, so one row per group gives the population. `field_total` covers bachelor's and graduate holders (ACS asks the field only of them); `degree_x_soc` covers all four degree levels.

In [2]:
pop = G[G.cut == "degree_x_soc"].drop_duplicates(["age_band", "degree_level"]).pivot(index="degree_level", columns="age_band", values="group_workers")
pop = pop.reindex(["below associate", "associate", "bachelor", "graduate"])[["22-27", "28-34", "22-34", "35-64"]]
show(pop.round(0).astype(int), "Employed, not-enrolled NYC residents by highest degree and age band (ACS 2020-2024 period average)")
ft = G[(G.cut == "field_total")]
nfield = ft.groupby("degree_level").group_1.nunique()
mapped = ft[ft.group_1_label.notna()].groupby(["degree_level", "age_band"]).workers.sum() / ft.groupby(["degree_level", "age_band"]).workers.sum()
print(f"fields in field_total: bachelor {nfield.get('bachelor')}, graduate {nfield.get('graduate')}; the 78 mapped fields (ref_fod_cip) cover "
      f"{mapped.loc[('bachelor', '22-34')] * 100:.1f}% of bachelor's workers 22-34 and {mapped.loc[('bachelor', '35-64')] * 100:.1f}% at 35-64 (unsuppressed rows)")

**Employed, not-enrolled NYC residents by highest degree and age band (ACS 2020-2024 period average)**

age_band,22-27,28-34,22-34,35-64
degree_level,,,,
below associate,153698,226035,379733,1070582
associate,19451,41639,61090,152620
bachelor,203282,285750,489032,553485
graduate,45310,179824,225134,477351


fields in field_total: bachelor 173, graduate 169; the 78 mapped fields (ref_fod_cip) cover 98.1% of bachelor's workers 22-34 and 95.8% at 35-64 (unsuppressed rows)


## 2. Where young bachelor's holders work (`degree_x_soc`)
The landed job is the Census occupation code mapped to `dim_job`: 474 detailed codes plus 67 broad or minor groups where the Census code is itself a group. `includes_allocated_codes` marks cells reached through a split Census group (e.g. Managers All Other is the 1191XX mask minus the codes PUMS publishes on their own).

In [3]:
D = G[(G.cut == "degree_x_soc")].merge(jobs, left_on="group_2", right_on="soc_code", how="left")
assert D.soc_code.notna().all(), "every landed code must be in dim_job"
b = D[(D.degree_level == "bachelor") & (D.age_band == "22-34")].copy()
print(f"bachelor's 22-34: {len(b)} landed jobs, {int((~b.suppressed).sum())} with 30+ responses; group_workers {b.group_workers.iloc[0]:,.0f}; "
      f"unsuppressed rows carry {b.workers.sum() / b.group_workers.iloc[0] * 100:.1f}% of workers")
cols = ["group_2", "group_2_label", "soc_level", "entry_education", "is_degree_entry", "n_unweighted", "workers", "moe_workers", "share_pct", "includes_allocated_codes"]
b["share_pct"] = b.share_of_group * 100
show(b.nlargest(25, "workers")[cols].round(1).reset_index(drop=True), "The 25 jobs that employ the most bachelor's holders aged 22-34")


def entry_mix(d):
    """Share of the band's workers by the landed job's BLS entry education, two ways: weighted over unsuppressed rows, and unweighted person rows over every row."""
    d = d.copy()
    d["entry"] = np.select([d.entry_education_rank.isna(), d.entry_education_rank < 5, d.entry_education_rank == 5, d.entry_education_rank == 6, d.entry_education_rank > 6],
                           ["unknown (group code)", "below associate's", "associate's", "bachelor's", "graduate"], default="unknown (group code)")
    w = d.groupby("entry").workers.sum() / d.workers.sum() * 100
    u = d.groupby("entry").n_unweighted.sum() / d.n_unweighted.sum() * 100
    out = pd.DataFrame({"weighted, unsuppressed rows %": w, "unweighted, all rows %": u}).reindex(["below associate's", "associate's", "bachelor's", "graduate", "unknown (group code)"])
    return out


for band in ["22-27", "22-34", "35-64"]:
    d = D[(D.degree_level == "bachelor") & (D.age_band == band)]
    t = entry_mix(d)
    under = t.loc[["below associate's", "associate's"], "weighted, unsuppressed rows %"].sum()
    under_u = t.loc[["below associate's", "associate's"], "unweighted, all rows %"].sum()
    show(t.round(1), f"Bachelor's holders {band}: BLS typical entry education of the job they hold (underemployed = below a bachelor's: {under:.1f}% weighted, {under_u:.1f}% unweighted)")
deg = b.groupby("is_degree_entry").workers.sum() / b.workers.sum() * 100
print(f"bachelor's 22-34 in degree-reachable jobs (is_degree_entry): {deg.get(True, 0):.1f}% of unsuppressed workers")

bachelor's 22-34: 541 landed jobs, 152 with 30+ responses; group_workers 489,032; unsuppressed rows carry 86.2% of workers


**The 25 jobs that employ the most bachelor's holders aged 22-34**

,group_2,group_2_label,soc_level,entry_education,is_degree_entry,n_unweighted,workers,moe_workers,share_pct,includes_allocated_codes
0,15-1252,Software Developers,detailed,Bachelor's degree,True,1052,"25,228.0","1,393.6",5.2,False
1,11-9199,"Managers, All Other",detailed,Bachelor's degree,True,801,"19,566.6","1,697.7",4.0,True
2,29-1141,Registered Nurses,detailed,Bachelor's degree,True,528,"13,326.0","1,388.8",2.7,False
3,13-1111,Management Analysts,detailed,Bachelor's degree,True,424,"11,506.0","1,259.6",2.4,False
4,13-2011,Accountants and Auditors,detailed,Bachelor's degree,True,474,"11,131.0","1,301.6",2.3,False
5,13-1161,Market Research Analysts and Marketing Specialists,detailed,Bachelor's degree,True,369,"10,728.0","1,410.2",2.2,False
6,11-2021,Marketing Managers,detailed,Bachelor's degree,True,408,"10,316.0","1,281.5",2.1,False
7,13-2051,Financial and Investment Analysts,detailed,Bachelor's degree,True,384,"10,235.0","1,346.8",2.1,False
8,43-4051,Customer Service Representatives,detailed,High school diploma or equivalent,False,344,"9,107.0","1,076.9",1.9,False
9,11-3031,Financial Managers,detailed,Bachelor's degree,True,341,"9,064.0","1,130.4",1.9,False


**Bachelor's holders 22-27: BLS typical entry education of the job they hold (underemployed = below a bachelor's: 23.4% weighted, 36.5% unweighted)**

,"weighted, unsuppressed rows %","unweighted, all rows %"
entry,,
below associate's,21.4,29.8
associate's,2.0,6.7
bachelor's,61.0,46.9
graduate,0.2,4.4
unknown (group code),15.5,12.1


**Bachelor's holders 22-34: BLS typical entry education of the job they hold (underemployed = below a bachelor's: 24.7% weighted, 36.6% unweighted)**

,"weighted, unsuppressed rows %","unweighted, all rows %"
entry,,
below associate's,23.3,29.9
associate's,1.4,6.7
bachelor's,58.7,47.0
graduate,1.0,4.2
unknown (group code),15.6,12.2


**Bachelor's holders 35-64: BLS typical entry education of the job they hold (underemployed = below a bachelor's: 28.0% weighted, 40.1% unweighted)**

,"weighted, unsuppressed rows %","unweighted, all rows %"
entry,,
below associate's,26.6,34.0
associate's,1.4,6.1
bachelor's,50.6,40.7
graduate,2.0,3.8
unknown (group code),19.3,15.4


bachelor's 22-34 in degree-reachable jobs (is_degree_entry): 60.2% of unsuppressed workers


In [4]:
a = D[(D.degree_level == "associate") & (D.age_band == "22-34")].copy()
a["share_pct"] = a.share_of_group * 100
print(f"associate's 22-34: {len(a)} landed jobs, {int((~a.suppressed).sum())} with 30+ responses; group_workers {a.group_workers.iloc[0]:,.0f}")
show(a.nlargest(15, "workers")[cols].round(1).reset_index(drop=True), "The 15 jobs that employ the most associate's holders aged 22-34")
ta = entry_mix(a)
show(ta.round(1), "Associate's holders 22-34: entry education of the job they hold")
dega = a.groupby("is_degree_entry").workers.sum() / a.workers.sum() * 100
print(f"associate's 22-34 in degree-reachable jobs: {dega.get(True, 0):.1f}% of unsuppressed workers")

associate's 22-34: 362 landed jobs, 18 with 30+ responses; group_workers 61,090


**The 15 jobs that employ the most associate's holders aged 22-34**

,group_2,group_2_label,soc_level,entry_education,is_degree_entry,n_unweighted,workers,moe_workers,share_pct,includes_allocated_codes
0,33-3050,Police Officers,broad,NaN,False,48,"1,754.0",567.1,2.9,False
1,41-2031,Retail Salespersons,detailed,No formal educational credential,False,70,"1,720.0",463.5,2.8,False
2,25-9040,Teaching Assistants,broad,NaN,False,45,"1,585.0",456.9,2.6,False
3,33-9030,Security Guards and Gambling Surveillance Officers,broad,NaN,False,39,"1,526.0",586.8,2.5,False
4,31-1121,Home Health Aides,detailed,NaN,False,49,"1,455.0",494.4,2.4,False
5,43-4051,Customer Service Representatives,detailed,High school diploma or equivalent,False,58,"1,434.0",402.3,2.3,False
6,29-1141,Registered Nurses,detailed,Bachelor's degree,True,49,"1,383.0",376.4,2.3,False
7,43-4171,Receptionists and Information Clerks,detailed,High school diploma or equivalent,False,53,"1,318.0",396.2,2.2,False
8,41-2010,Cashiers,broad,NaN,False,49,"1,257.0",462.1,2.1,False
9,43-9061,"Office Clerks, General",detailed,High school diploma or equivalent,False,38,"1,235.0",494.2,2.0,False


**Associate's holders 22-34: entry education of the job they hold**

,"weighted, unsuppressed rows %","unweighted, all rows %"
entry,,
below associate's,46.5,58.0
associate's,0.1,5.9
bachelor's,10.9,15.9
graduate,0.0,1.2
unknown (group code),42.5,18.9


associate's 22-34 in degree-reachable jobs: 11.0% of unsuppressed workers


## 3. What each field's graduates do (`field_x_soc_major`, bachelor's 22-34)
Shares over the 22 SOC major groups per field. Dispersion = share of the field's workers in its three largest groups (over unsuppressed rows; the suppressed remainder is listed).

In [5]:
F = G[(G.cut == "field_x_soc_major") & (G.degree_level == "bachelor") & (G.age_band == "22-34")].copy()
F["share_pct"] = F.share_of_group * 100
# All-bachelor's mix by major group, from the degree_x_soc cut: weighted over unsuppressed rows (86% of workers, renormalized) and
# unweighted person rows over every row (100% of responses). The lift in §4b uses the weighted pair where the field cell is readable, the unweighted pair where it is suppressed.
allb = b.groupby("major_code").workers.sum() / b.workers.sum() * 100
allb_u = b.groupby("major_code").n_unweighted.sum() / b.n_unweighted.sum() * 100
allb.name = "all bachelor's 22-34 %"
mg_labels = jobs[jobs.soc_level == "major"].set_index("soc_code").soc_title.str.replace(" Occupations", "")
rows = []
for f, d in F.groupby("group_1"):
    d = d.sort_values("workers", ascending=False)
    top = d[~d.suppressed].head(3)
    rows.append({"field": f, "label": d.group_1_label.iloc[0], "workers": d.group_workers.iloc[0], "n": d.n_unweighted.sum(),
                 "top 3 groups": "; ".join(f"{mg_labels.get(g, g)} {s:.0f}%" for g, s in zip(top.group_2, top.share_pct)),
                 "top-3 share %": top.share_pct.sum(), "suppressed remainder %": 100 - d.share_pct.sum(), "groups with 30+": int((~d.suppressed).sum())})
FD = pd.DataFrame(rows).sort_values("workers", ascending=False)
show(FD[FD.label.notna()].head(20).round(1).reset_index(drop=True), "The 20 largest mapped fields: where their bachelor's-only holders aged 22-34 work")
show(FD[FD.field.isin(TEST)].round(1).reset_index(drop=True), "The three D16 test fields")
md("**All bachelor's holders 22-34 by major group** (the comparator for §4): " + "; ".join(f"{mg_labels.get(g, g)} {s:.1f}%" for g, s in allb.sort_values(ascending=False).head(8).items()))

**The 20 largest mapped fields: where their bachelor's-only holders aged 22-34 work**

,field,label,workers,n,top 3 groups,top-3 share %,suppressed remainder %,groups with 30+
0,5501,Economics,"27,558.0",990,Business and Financial Operations 34%; Management 20%; S...,65.0,13.1,6
1,6207,Finance,"25,408.0",902,Business and Financial Operations 32%; Management 24%; S...,69.4,12.2,5
2,2102,Computer Science,"24,169.0",988,Computer and Mathematical 57%; Business and Financial Op...,73.5,18.5,5
3,5200,Psychology,"21,126.0",850,Management 17%; Office and Administrative Support 15%; B...,44.1,11.9,10
4,6203,Business Management And Administration,"19,539.0",757,Management 23%; Business and Financial Operations 17%; O...,55.0,22.5,6
5,6200,General Business,"19,496.0",677,Business and Financial Operations 25%; Management 18%; S...,60.1,22.1,5
6,1901,Communications,"19,449.0",670,Management 23%; Business and Financial Operations 23%; A...,63.3,17.6,5
7,6206,Marketing,"17,787.0",692,Management 29%; Business and Financial Operations 21%; S...,64.9,13.9,6
8,6201,Accounting,"14,926.0",611,Business and Financial Operations 47%; Management 13%; O...,68.5,19.2,5
9,3301,English Language And Literature,"14,468.0",639,"Arts, Design, Entertainment, Sports, and Media 20%; Mana...",49.1,16.4,7


**The three D16 test fields**

,field,label,workers,n,top 3 groups,top-3 share %,suppressed remainder %,groups with 30+
0,5200,Psychology,"21,126.0",850,Management 17%; Office and Administrative Support 15%; B...,44.1,11.9,10
1,5506,Political Science And Government,"13,938.0",531,Business and Financial Operations 23%; Management 17%; A...,50.4,24.1,6
2,5301,Criminal Justice And Fire Protection,"8,867.0",356,Protective Service 16%; Office and Administrative Suppor...,42.2,57.8,3


**All bachelor's holders 22-34 by major group** (the comparator for §4): Management 18.3%; Business and Financial Operations 18.3%; Arts, Design, Entertainment, Sports, and Media 11.5%; Computer and Mathematical 11.1%; Office and Administrative Support 9.8%; Sales and Related 9.7%; Educational Instruction and Library 5.4%; Healthcare Practitioners and Technical 3.6%

## 4. The D16 test
### 4a. Lift per field: linked share ÷ all-bachelor's base rate (§3e)
The linked set is rebuilt here exactly as `int_grad_outcome_person_cut` builds it: bachelor's rows of `bridge_major_job` whose CIP is, or is under, one of the field's `ref_fod_cip` codes. The base rate sums `degree_x_soc` shares over those detailed jobs (suppressed rows count 0 and are counted in `linked jobs suppressed`).

In [6]:
fod = q("select fod1p, fod_title, cip2020 from ref.ref_fod_cip")
bridge = q("select cip_code, soc_code, is_all_other, weight_openings, openings_nyc from bridge_major_job where award_level = 5")
pairs = fod.merge(bridge, how="cross")
pairs = pairs[[c.startswith(p) for c, p in zip(pairs.cip_code, pairs.cip2020)]][["fod1p", "soc_code", "is_all_other"]].drop_duplicates(["fod1p", "soc_code"])
print(f"linked (field, job) pairs rebuilt: {len(pairs):,} over {pairs.fod1p.nunique()} fields and {pairs.soc_code.nunique()} jobs")
bs = b.set_index("group_2")
L = G[(G.cut == "field_x_linked") & (G.degree_level == "bachelor") & (G.age_band == "22-34")].pivot(index="group_1", columns="group_2", values="share_of_group") * 100
Ln = G[(G.cut == "field_x_linked") & (G.degree_level == "bachelor") & (G.age_band == "22-34") & (G.group_2 == "linked")].set_index("group_1")[["n_unweighted", "group_workers", "moe_share", "group_1_label"]]
rows = []
for f, d in pairs.groupby("fod1p"):
    inb = bs.reindex(d.soc_code)
    rows.append({"field": f, "label": Ln.group_1_label.get(f), "workers": Ln.group_workers.get(f), "linked jobs": len(d), "all-other jobs": int(d.is_all_other.sum()),
                 "linked share %": L.loc[f, "linked"] if f in L.index else np.nan, "moe": Ln.moe_share.get(f, np.nan) * 100, "n linked": Ln.n_unweighted.get(f),
                 "not detailed %": L.loc[f, "job_not_detailed"] if f in L.index else np.nan,
                 "base rate %": inb.share_of_group.fillna(0).sum() * 100, "linked jobs suppressed": int(inb.suppressed.fillna(True).sum()),
                 "top linked job": inb.workers.idxmax() if inb.workers.notna().any() else None})
T = pd.DataFrame(rows)
T["lift"] = T["linked share %"] / T["base rate %"]
T["verdict"] = np.select([T.lift.isna(), T.lift >= 3, T.lift >= 1.5], ["n < 30", "pathway", "weak"], "convention")
T["top linked job"] = T["top linked job"].map(jobs.set_index("soc_code").soc_title)
T = T.sort_values("workers", ascending=False)
show(T.verdict.value_counts().rename("fields").to_frame().T, "Verdict counts over the 78 mapped fields (bachelor's 22-34)")
tcols = ["field", "label", "workers", "linked jobs", "all-other jobs", "linked share %", "moe", "n linked", "base rate %", "lift", "verdict", "top linked job", "not detailed %"]
show(T[T["linked share %"].notna()].sort_values("lift", ascending=False)[tcols].round(1).reset_index(drop=True), "Every field with a readable linked share (n ≥ 30), ranked by lift")
show(T[T.field.isin(TEST)][tcols].round(1).reset_index(drop=True), "The three D16 test fields")
print(f"fields whose linked share is suppressed (fewer than 30 of the field's workers hold any linked job): {int(T['linked share %'].isna().sum())}, "
      f"holding {T.loc[T['linked share %'].isna(), 'workers'].sum():,.0f} workers")

linked (field, job) pairs rebuilt: 649 over 75 fields and 291 jobs


**Verdict counts over the 78 mapped fields (bachelor's 22-34)**

verdict,n < 30,pathway,convention,weak
fields,51,17,5,2


**Every field with a readable linked share (n ≥ 30), ranked by lift**

,field,label,workers,linked jobs,all-other jobs,linked share %,moe,n linked,base rate %,lift,verdict,top linked job,not detailed %
0,1401,Architecture,"4,392.0",3,0,29.6,6.1,54.0,0.5,57.1,pathway,"Architects, Except Landscape and Naval",NaN
1,2406,Civil Engineering,"2,383.0",9,1,64.9,13.1,51.0,1.5,42.2,pathway,Civil Engineers,NaN
2,6002,Music,"6,316.0",5,0,11.2,3.6,38.0,0.3,36.7,pathway,Musicians and Singers,14.4
3,6107,Nursing,"12,133.0",2,0,73.9,3.5,363.0,3.2,23.3,pathway,Registered Nurses,NaN
4,6004,Commercial Art And Graphic Design,"13,899.0",15,2,40.3,4.5,259.0,3.5,11.6,pathway,Graphic Designers,12.4
5,6005,Film Video And Photographic Arts,"8,714.0",4,0,17.1,5.0,57.0,1.7,9.9,pathway,Producers and Directors,22.1
6,2407,Computer Engineering,"4,553.0",9,1,56.2,9.0,99.0,5.8,9.7,pathway,Software Developers,NaN
7,2105,Information Sciences,"2,119.0",6,1,46.1,13.2,32.0,6.5,7.1,pathway,Software Developers,NaN
8,6201,Accounting,"14,926.0",13,0,46.6,5.1,301.0,6.7,6.9,pathway,Accountants and Auditors,8.4
9,2102,Computer Science,"24,169.0",16,1,52.7,3.4,531.0,8.8,6.0,pathway,Software Developers,8.3


**The three D16 test fields**

,field,label,workers,linked jobs,all-other jobs,linked share %,moe,n linked,base rate %,lift,verdict,top linked job,not detailed %
0,5200,Psychology,"21,126.0",7,3,4.3,1.9,47.0,4.2,1.0,convention,"Managers, All Other",20.5
1,5506,Political Science And Government,"13,938.0",5,1,4.9,1.6,35.0,4.1,1.2,convention,"Managers, All Other",11.0
2,5301,Criminal Justice And Fire Protection,"8,867.0",27,4,10.9,2.7,40.0,7.7,1.4,convention,"Managers, All Other",25.2


fields whose linked share is suppressed (fewer than 30 of the field's workers hold any linked job): 51, holding 114,452 workers


### 4b. The test fields by job family against the all-bachelor's mix (§3e rule: 1.5× = affinity)
Option A (All Other weight) is supported where Management (`11-0000`) runs at 1.5× the all-bachelor's share; option B where the named job's family does (`19-0000` Life, Physical and Social Science for Psychology and Political Science; `33-0000` Protective Service for Criminal Justice).

In [7]:
rows = []
for f, name in TEST.items():
    d = F[F.group_1 == f].set_index("group_2")
    for g in ["11-0000", "19-0000", "33-0000", "43-0000", "13-0000", "25-0000", "21-0000", "29-0000", "41-0000"]:
        r = d.loc[g] if g in d.index else None
        share = r.share_pct if r is not None and not r.suppressed else np.nan
        ub = (r.n_unweighted / d.n_unweighted.sum() * 100) if r is not None else np.nan
        rows.append({"field": name, "group": f"{g} {mg_labels.get(g, g)}", "field share %": share, "moe": (r.moe_share * 100) if r is not None and not r.suppressed else np.nan,
                     "n": r.n_unweighted if r is not None else 0, "unweighted share %": ub, "all bachelor's % (weighted)": allb.get(g, np.nan), "all bachelor's % (unweighted)": allb_u.get(g, np.nan),
                     "lift": (share / allb.get(g, np.nan)) if pd.notna(share) else (ub / allb_u.get(g, np.nan))})
X = pd.DataFrame(rows)
show(X.round(2), "Test fields: share of the field's bachelor's-only workers 22-34 in each job family vs all bachelor's holders (where the field cell is suppressed, a null field share, the unweighted pair gives the lift)")
for f, name in TEST.items():
    x = X[X.field == name].set_index("group")
    mgmt = x.loc["11-0000 Management", "lift"]
    named = x.loc["33-0000 Protective Service" if f == "5301" else "19-0000 Life, Physical, and Social Science", "lift"]
    verdict = "A supported" if mgmt >= 1.5 else ("B supported" if named >= 1.5 else "neither: both weights are conventions")
    print(f"{name}: management lift {mgmt:.2f}, named-job family lift {named:.2f} → {verdict}")

**Test fields: share of the field's bachelor's-only workers 22-34 in each job family vs all bachelor's holders (where the field cell is suppressed, a null field share, the unweighted pair gives the lift)**

,field,group,field share %,moe,n,unweighted share %,all bachelor's % (weighted),all bachelor's % (unweighted),lift
0,Psychology,11-0000 Management,17.0,3.2,135,15.9,18.3,23.0,0.9
1,Psychology,"19-0000 Life, Physical, and Social Science",NaN,NaN,19,2.2,0.8,2.3,1.0
2,Psychology,33-0000 Protective Service,NaN,NaN,13,1.5,1.2,1.2,1.2
3,Psychology,43-0000 Office and Administrative Support,15.2,3.3,127,14.9,9.8,8.7,1.6
4,Psychology,13-0000 Business and Financial Operations,11.9,2.0,116,13.7,18.3,12.4,0.7
5,Psychology,25-0000 Educational Instruction and Library,9.1,1.9,70,8.2,5.4,5.6,1.7
6,Psychology,21-0000 Community and Social Service,5.5,1.3,45,5.3,1.5,1.4,3.8
7,Psychology,29-0000 Healthcare Practitioners and Technical,7.8,2.0,56,6.6,3.6,3.9,2.2
8,Psychology,41-0000 Sales and Related,7.7,1.9,72,8.5,9.7,6.4,0.8
9,Political Science And Government,11-0000 Management,17.0,2.7,96,18.1,18.3,23.0,0.9


Psychology: management lift 0.93, named-job family lift 0.99 → neither: both weights are conventions
Political Science And Government: management lift 0.93, named-job family lift 0.92 → neither: both weights are conventions
Criminal Justice And Fire Protection: management lift 0.58, named-job family lift 13.11 → B supported


### 4c. Does the picture change with age?
The same linked share at 22-27, 28-34 and 35-64 for the test and contrast fields (suppressed cells shown as null).

In [8]:
Lb = G[(G.cut == "field_x_linked") & (G.degree_level == "bachelor") & (G.group_2 == "linked") & G.group_1.isin(list(TEST) + CONTRAST)]
age = Lb.pivot(index="group_1_label", columns="age_band", values="share_of_group")[["22-27", "28-34", "22-34", "35-64"]] * 100
show(age.round(1), "Linked share by age band, bachelor's-only holders (%)")

**Linked share by age band, bachelor's-only holders (%)**

age_band,22-27,28-34,22-34,35-64
group_1_label,,,,
Accounting,51.4,44.2,46.6,33.5
Biology,NaN,NaN,NaN,NaN
Business Management And Administration,20.7,30.3,26.4,26.8
Computer Science,57.3,47.4,52.7,35.7
Criminal Justice And Fire Protection,NaN,NaN,10.9,14.9
Economics,28.9,17.3,22.9,9.7
English Language And Literature,NaN,9.4,8.9,8.5
Finance,45.6,37.7,41.2,27.6
Nursing,71.5,75.2,73.9,68.4


## 5. Earnings and the graduate-degree route
Median earnings of full-time year-round workers in 2024 dollars, 90% MOE from replicate weights; null where fewer than 30 full-time year-round responses.

In [9]:
E = ft[(ft.degree_level == "bachelor") & (ft.age_band == "22-34") & ft.group_1_label.notna() & ~ft.earnings_suppressed].copy()
E["median"] = E.median_earnings_2024usd
E["moe"] = E.moe_median_earnings
ecols = ["group_1", "group_1_label", "workers", "n_ftyr", "median", "moe", "mean_earnings_2024usd"]
show(E.nlargest(12, "median")[ecols].round(0).reset_index(drop=True), "Highest-earning fields, bachelor's-only holders 22-34 (median, full-time year-round)")
show(E.nsmallest(12, "median")[ecols].round(0).reset_index(drop=True), "Lowest-earning fields")
allmed = ft[(ft.degree_level == "bachelor") & (ft.age_band == "22-34")]
tot = (allmed.workers * allmed.median_earnings_2024usd).sum() / allmed.loc[allmed.median_earnings_2024usd.notna(), "workers"].sum()
print(f"workers-weighted mean of field medians, bachelor's 22-34: ${tot:,.0f}; fields with a readable median: {len(E)} of 78 mapped")

**Highest-earning fields, bachelor's-only holders 22-34 (median, full-time year-round)**

,group_1,group_1_label,workers,n_ftyr,median,moe,mean_earnings_2024usd
0,3701,Applied Mathematics,"1,098.0",35,"178,751.0","37,502.0","222,718.0"
1,2407,Computer Engineering,"4,553.0",155,"140,532.0","14,914.0","171,067.0"
2,2105,Information Sciences,"2,119.0",67,"126,906.0","27,320.0","147,401.0"
3,5402,Public Policy,"1,410.0",41,"122,904.0","84,036.0","152,804.0"
4,3700,Mathematics,"6,016.0",208,"122,202.0","11,553.0","165,358.0"
5,2102,Computer Science,"24,169.0",800,"122,202.0","5,657.0","156,366.0"
6,6207,Finance,"25,408.0",774,"122,202.0","4,995.0","171,212.0"
7,5501,Economics,"27,558.0",847,"121,830.0","4,096.0","165,190.0"
8,6212,Management Information Systems And Statistics,"1,542.0",52,"115,442.0","20,986.0","165,449.0"
9,5007,Physics,"2,081.0",65,"107,392.0","11,625.0","120,099.0"


**Lowest-earning fields**

,group_1,group_1_label,workers,n_ftyr,median,moe,mean_earnings_2024usd
0,5301,Criminal Justice And Fire Protection,"8,867.0",266,"58,657.0","3,841.0","89,971.0"
1,5404,Social Work,"1,967.0",45,"58,770.0","10,948.0","74,239.0"
2,6109,Treatment Therapy Professions,"1,785.0",54,"59,662.0","22,730.0","73,068.0"
3,2304,Elementary Education,908.0,31,"60,328.0","2,441.0","76,565.0"
4,2300,General Education,"4,640.0",112,"61,467.0","7,971.0","72,541.0"
5,3611,Neuroscience,"1,602.0",51,"62,968.0","23,678.0","99,197.0"
6,6100,General Medical And Health Services,"2,411.0",88,"63,545.0","23,213.0","105,128.0"
7,1301,Environmental Science,"2,121.0",62,"65,628.0","8,761.0","89,276.0"
8,3601,Biochemical Sciences,"1,346.0",44,"65,991.0","21,072.0","78,894.0"
9,6001,Drama And Theater Arts,"12,104.0",301,"67,211.0","8,992.0","79,531.0"


workers-weighted mean of field medians, bachelor's 22-34: $90,249; fields with a readable median: 72 of 78 mapped


In [10]:
LE = G[(G.cut == "field_x_linked") & (G.degree_level == "bachelor") & (G.age_band == "22-34") & ~G.earnings_suppressed]
pe = LE.pivot(index="group_1_label", columns="group_2", values="median_earnings_2024usd")
pe = pe[pe.get("linked").notna() & pe.get("not_linked").notna()] if "linked" in pe else pe.iloc[0:0]
pe["linked premium %"] = (pe.linked / pe.not_linked - 1) * 100
show(pe.sort_values("linked premium %", ascending=False).round(0), "Median earnings in a linked job vs not, where both are readable (bachelor's 22-34)")

**Median earnings in a linked job vs not, where both are readable (bachelor's 22-34)**

group_2,job_not_detailed,linked,not_linked,linked premium %
group_1_label,,,,
Computer Engineering,NaN,"172,592.0","76,144.0",127.0
Computer Science,"60,915.0","157,420.0","94,452.0",67.0
Nursing,NaN,"102,649.0","67,166.0",53.0
Finance,"94,961.0","158,862.0","104,571.0",52.0
Computer And Information Systems-General,NaN,"100,547.0","71,068.0",41.0
Psychology,"67,032.0","106,601.0","75,969.0",40.0
Mass Media,"79,947.0","84,907.0","62,968.0",35.0
Film Video And Photographic Arts,"73,463.0","82,908.0","65,991.0",26.0
Business Management And Administration,"73,098.0","101,525.0","83,789.0",21.0


In [11]:
g35 = ft[(ft.age_band == "35-64") & ft.group_1_label.notna()]
gr = g35.pivot(index="group_1", columns="degree_level", values="workers")          # null where the level's cell is suppressed
gn = g35.pivot(index="group_1", columns="degree_level", values="n_unweighted")
gr["graduate share %"] = gr.graduate / (gr.bachelor + gr.graduate) * 100        # null unless both cells are readable
gr["graduate share % (unweighted, all rows)"] = gn.graduate / (gn.bachelor + gn.graduate) * 100
gr = gr.join(g35.drop_duplicates("group_1").set_index("group_1").group_1_label).reset_index()
print(f"fields with both the bachelor's and the graduate cell readable at 35-64: {int(gr['graduate share %'].notna().sum())} of {len(gr)}")
show(gr.nlargest(12, "graduate share %").round(1).reset_index(drop=True), "Fields whose holders most often went on to a graduate degree (35-64, highest degree graduate ÷ bachelor's + graduate)")
show(gr[gr.group_1.isin(list(TEST) + CONTRAST)].round(1).reset_index(drop=True), "Test and contrast fields")
allgr = pop.loc["graduate", "35-64"] / (pop.loc["bachelor", "35-64"] + pop.loc["graduate", "35-64"]) * 100
print(f"all fields: {allgr:.1f}% of bachelor's-or-higher holders aged 35-64 hold a graduate degree")

fields with both the bachelor's and the graduate cell readable at 35-64: 66 of 78


**Fields whose holders most often went on to a graduate degree (35-64, highest degree graduate ÷ bachelor's + graduate)**

,group_1,bachelor,graduate,graduate share %,"graduate share % (unweighted, all rows)",group_1_label
0,3601,696.0,"2,223.0",76.2,75.4,Biochemical Sciences
1,2304,"2,154.0","6,642.0",75.5,76.2,Elementary Education
2,3600,"5,532.0","16,918.0",75.4,76.2,Biology
3,5007,"1,474.0","4,037.0",73.3,74.6,Physics
4,5003,"2,190.0","4,884.0",69.0,69.0,Chemistry
5,6402,"9,969.0","19,429.0",66.1,66.0,History
6,5502,"2,030.0","3,685.0",64.5,59.5,Anthropology And Archeology
7,2405,982.0,"1,762.0",64.2,62.6,Chemical Engineering
8,1501,"2,364.0","4,184.0",63.9,63.9,Area Ethnic And Civilization Studies
9,5506,"12,316.0","21,515.0",63.6,63.7,Political Science And Government


**Test and contrast fields**

,group_1,bachelor,graduate,graduate share %,"graduate share % (unweighted, all rows)",group_1_label
0,2102,"19,557.0","9,869.0",33.5,33.4,Computer Science
1,3301,"20,371.0","27,255.0",57.2,56.0,English Language And Literature
2,3600,"5,532.0","16,918.0",75.4,76.2,Biology
3,5200,"21,242.0","27,474.0",56.4,56.7,Psychology
4,5301,"7,883.0","3,722.0",32.1,30.0,Criminal Justice And Fire Protection
5,5501,"24,391.0","23,853.0",49.4,51.2,Economics
6,5506,"12,316.0","21,515.0",63.6,63.7,Political Science And Government
7,6107,"24,260.0","11,728.0",32.6,34.4,Nursing
8,6201,"35,415.0","13,428.0",27.5,27.0,Accounting
9,6203,"31,373.0","14,936.0",32.3,30.5,Business Management And Administration


all fields: 46.3% of bachelor's-or-higher holders aged 35-64 hold a graduate degree


### 5c. Does the graduate degree pay off, by field? (added; rule: plan §3h, written before this ran)
`field_total`, highest degree bachelor's vs graduate, the bachelor's field in both cases (ACS asks only that; the graduate field is unknown). Bands 28–34 and 35–64 (the 22–27 graduate cells are mostly suppressed). Premium = graduate median ÷ bachelor's median − 1 (full-time year-round, 2024 $), the mean premium beside it. Readable when the Census two-estimate test passes: SE = MOE ÷ 1.645, Z = difference ÷ √(SE_g² + SE_b²), |Z| > 1.645; otherwise "no readable difference". A premium, not a return: selection into graduate study and the cost of it are not counted.

In [12]:
Z90 = 1.645


def payoff(band: str) -> pd.DataFrame:
    d = ft[(ft.age_band == band) & ft.group_1_label.notna()]
    p = d.pivot(index="group_1", columns="degree_level", values=["median_earnings_2024usd", "moe_median_earnings", "mean_earnings_2024usd", "workers", "n_ftyr"])
    p.columns = [f"{a}_{b}" for a, b in p.columns]
    p = p.rename(columns={"median_earnings_2024usd_bachelor": "median_b", "median_earnings_2024usd_graduate": "median_g", "moe_median_earnings_bachelor": "moe_b",
                          "moe_median_earnings_graduate": "moe_g", "mean_earnings_2024usd_bachelor": "mean_b", "mean_earnings_2024usd_graduate": "mean_g",
                          "workers_bachelor": "workers_b", "workers_graduate": "workers_g", "n_ftyr_bachelor": "n_b", "n_ftyr_graduate": "n_g"})
    p["premium %"] = (p.median_g / p.median_b - 1) * 100
    p["mean premium %"] = (p.mean_g / p.mean_b - 1) * 100
    p["z"] = (p.median_g - p.median_b) / np.sqrt((p.moe_g / Z90) ** 2 + (p.moe_b / Z90) ** 2)
    p["verdict"] = np.select([p.median_g.isna() | p.median_b.isna(), p.z.abs() > Z90], ["unreadable (a median suppressed)", "readable"], default="no readable difference")
    p["graduate share %"] = p.workers_g / (p.workers_b + p.workers_g) * 100
    p = p.join(d.drop_duplicates("group_1").set_index("group_1").group_1_label.rename("field"))
    p["band"] = band
    return p.reset_index()


PAY = pd.concat([payoff("28-34"), payoff("35-64")], ignore_index=True)
pcols = ["group_1", "field", "n_b", "n_g", "median_b", "moe_b", "median_g", "moe_g", "premium %", "z", "verdict", "mean premium %", "graduate share %"]
for band in ["28-34", "35-64"]:
    pb = PAY[PAY.band == band]
    vc = pb.verdict.value_counts()
    readable = pb[pb.verdict == "readable"]
    print(f"{band}: {len(pb)} mapped fields → readable {vc.get('readable', 0)}, no readable difference {vc.get('no readable difference', 0)}, "
          f"unreadable {vc.get('unreadable (a median suppressed)', 0)}; among readable: median premium {readable['premium %'].median():.0f}%, "
          f"negative {int((readable['premium %'] < 0).sum())}; mean/median premium sign disagreements {int((np.sign(readable['premium %']) != np.sign(readable['mean premium %'])).sum())}")
    show(readable.sort_values("premium %", ascending=False)[pcols].round(1).reset_index(drop=True), f"Fields with a readable graduate premium, {band} (median full-time year-round earnings, 2024 $, 90% MOE)")
    show(pb[pb.verdict == "no readable difference"].sort_values("workers_b", ascending=False)[pcols].round(1).reset_index(drop=True), f"No readable difference, {band}")
    show(pb[pb.verdict.str.startswith("unreadable")][["group_1", "field", "n_b", "n_g", "workers_b", "workers_g"]].round(0).reset_index(drop=True), f"Unreadable (a median suppressed), {band}")

28-34: 77 mapped fields → readable 9, no readable difference 34, unreadable 34; among readable: median premium 36%, negative 0; mean/median premium sign disagreements 0


**Fields with a readable graduate premium, 28-34 (median full-time year-round earnings, 2024 $, 90% MOE)**

,group_1,field,n_b,n_g,median_b,moe_b,median_g,moe_g,premium %,z,verdict,mean premium %,graduate share %
0,6200,General Business,335.0,127.0,"103,899.0","12,305.6","152,288.0","25,861.0",46.6,2.8,readable,25.1,26.1
1,6100,General Medical And Health Services,49.0,45.0,"71,068.0","24,774.6","101,525.0","15,736.8",42.9,1.7,readable,21.1,58.2
2,5301,Criminal Justice And Fire Protection,167.0,58.0,"64,435.0","7,747.6","91,651.0","4,111.5",42.2,5.1,readable,43.8,25.2
3,6201,Accounting,365.0,247.0,"91,372.0","11,428.7","129,594.0","11,093.4",41.8,3.9,readable,24.9,42.5
4,2102,Computer Science,382.0,122.0,"122,891.0","9,883.3","167,579.0","23,896.6",36.4,2.8,readable,46.8,22.7
5,5098,Multi-Disciplinary Or General Science,67.0,44.0,"73,098.0","16,409.0","96,539.0","12,740.1",32.1,1.9,readable,47.9,34.7
6,2414,Mechanical Engineering,85.0,58.0,"103,556.0","9,336.6","131,184.0","10,850.2",26.7,3.2,readable,15.3,38.6
7,6207,Finance,462.0,159.0,"131,982.0","9,398.1","152,288.0","16,250.3",15.4,1.8,readable,0.1,25.8
8,3301,English Language And Literature,324.0,261.0,"83,958.0","4,964.3","94,418.0","5,252.5",12.5,2.4,readable,0.2,43.3


**No readable difference, 28-34**

,group_1,field,n_b,n_g,median_b,moe_b,median_g,moe_g,premium %,z,verdict,mean premium %,graduate share %
0,5501,Economics,486.0,281.0,"146,769.0","12,864.9","143,189.0","24,068.6",-2.4,-0.2,no readable difference,3.2,38.0
1,5200,Psychology,376.0,448.0,"83,789.0","7,755.9","91,372.0","8,216.0",9.1,1.1,no readable difference,8.3,54.2
2,6203,Business Management And Administration,381.0,149.0,"95,459.0","7,819.2","106,601.0","23,664.1",11.7,0.7,no readable difference,15.3,28.8
3,1901,Communications,346.0,76.0,"91,651.0","10,323.9","91,651.0","19,207.0",0.0,0.0,no readable difference,10.8,17.1
4,6206,Marketing,321.0,72.0,"107,046.0","10,798.5","120,689.0","34,677.5",12.7,0.6,no readable difference,6.7,18.6
5,6004,Commercial Art And Graphic Design,296.0,39.0,"83,958.0","4,408.4","95,459.0","12,668.8",13.7,1.4,no readable difference,1.7,15.0
6,5506,Political Science And Government,261.0,283.0,"107,392.0","13,613.1","119,324.0","24,635.3",11.1,0.7,no readable difference,7.3,53.7
7,6107,Nursing,254.0,80.0,"100,232.0","5,929.7","105,093.0","17,740.8",4.8,0.4,no readable difference,15.7,24.1
8,1903,Mass Media,180.0,33.0,"78,204.0","9,612.6","76,611.0","15,034.8",-2.0,-0.1,no readable difference,-6.5,13.2
9,3600,Biology,179.0,287.0,"77,561.0","11,190.6","89,375.0","6,252.9",15.2,1.5,no readable difference,26.1,61.5


...

,group_1,field,n_b,n_g,median_b,moe_b,median_g,moe_g,premium %,z,verdict,mean premium %,graduate share %
0,2304,Elementary Education,74.0,255.0,"51,391.0","7,304.5","104,947.0","6,276.4",104.2,9.1,readable,7.8,75.5
1,2405,Chemical Engineering,41.0,68.0,"73,321.0","29,359.9","145,235.0","64,685.5",98.1,1.7,readable,38.5,64.2
2,6212,Management Information Systems And Statistics,80.0,30.0,"109,485.0","17,984.9","203,050.0","71,975.1",85.5,2.1,readable,46.9,29.2
3,3600,Biology,195.0,617.0,"89,205.0","8,450.2","156,407.0","7,371.1",75.3,9.9,readable,100.8,75.4
4,6207,Finance,613.0,362.0,"122,202.0","13,457.7","202,851.0","27,017.2",66.0,4.4,readable,39.0,35.8
5,5007,Physics,45.0,150.0,"111,678.0","28,057.8","178,986.0","31,008.2",60.3,2.6,readable,49.6,73.3
6,2400,General Engineering,218.0,233.0,"89,205.0","14,511.1","139,649.0","34,318.5",56.5,2.2,readable,53.5,46.0
7,2408,Electrical Engineering,211.0,239.0,"109,982.0","21,106.9","171,082.0","17,456.4",55.6,3.7,readable,67.3,53.7
8,3700,Mathematics,195.0,279.0,"122,202.0","19,560.7","189,923.0","21,934.6",55.4,3.8,readable,36.5,60.0
9,5098,Multi-Disciplinary Or General Science,197.0,128.0,"92,353.0","11,406.8","143,189.0","27,772.8",55.0,2.8,readable,65.9,39.1


**No readable difference, 35-64**

,group_1,field,n_b,n_g,median_b,moe_b,median_g,moe_g,premium %,z,verdict,mean premium %,graduate share %
0,3301,English Language And Literature,637.0,829.0,"109,778.0","10,489.3","111,719.0","5,458.8",1.8,0.3,no readable difference,11.7,57.2
1,6206,Marketing,478.0,169.0,"111,719.0","14,542.9","103,556.0","8,561.5",-7.3,-0.8,no readable difference,-3.8,24.8
2,6000,Fine Arts,385.0,225.0,"89,375.0","6,566.5","95,459.0","11,077.7",6.8,0.8,no readable difference,7.3,35.4
3,6004,Commercial Art And Graphic Design,424.0,99.0,"91,372.0","12,167.0","95,459.0","14,226.8",4.5,0.4,no readable difference,-10.3,18.0
4,6001,Drama And Theater Arts,234.0,157.0,"102,619.0","8,037.7","98,650.0","8,788.5",-3.9,-0.5,no readable difference,-3.0,36.4
5,5301,Criminal Justice And Fire Protection,268.0,113.0,"90,686.0","10,204.5","101,525.0","12,984.9",12.0,1.1,no readable difference,34.1,32.1
6,1902,Journalism,209.0,99.0,"117,769.0","10,990.1","109,145.0","20,327.1",-7.3,-0.6,no readable difference,-0.4,29.1
7,1903,Mass Media,204.0,82.0,"108,368.0","13,260.6","111,719.0","16,381.2",3.1,0.3,no readable difference,-3.9,28.3
8,6005,Film Video And Photographic Arts,184.0,66.0,"100,547.0","20,054.4","115,738.0","26,074.6",15.1,0.8,no readable difference,22.7,24.5
9,1401,Architecture,189.0,271.0,"116,092.0","17,545.5","131,184.0","10,680.7",13.0,1.2,no readable difference,18.5,55.0


**Unreadable (a median suppressed), 35-64**

,group_1,field,n_b,n_g,workers_b,workers_g
0,1301,Environmental Science,19.0,33.0,NaN,"1,067.0"
1,1904,Advertising And Public Relations,92.0,21.0,"3,398.0",NaN
2,2001,Communication Technologies,43.0,8.0,"1,357.0",NaN
3,2106,Computer Administration Management And Security,16.0,11.0,NaN,NaN
4,3601,Biochemical Sciences,29.0,94.0,696.0,"2,223.0"
5,3608,Physiology,11.0,22.0,NaN,NaN
6,3611,Neuroscience,6.0,31.0,NaN,"1,035.0"
7,3701,Applied Mathematics,20.0,42.0,NaN,"1,178.0"
8,4009,Data Science and Data Analytics,1.0,0.0,NaN,NaN
9,5402,Public Policy,13.0,37.0,NaN,919.0


In [13]:
allf = ft[ft.median_earnings_2024usd.notna()]
ref = (allf.assign(wm=allf.workers * allf.median_earnings_2024usd).groupby(["age_band", "degree_level"]).agg(wm=("wm", "sum"), w=("workers", "sum"), fields=("group_1", "nunique")))
ref["workers-weighted mean of field medians"] = ref.wm / ref.w
ref = ref["workers-weighted mean of field medians"].unstack("degree_level").loc[["28-34", "35-64"]]
ref["premium %"] = (ref.graduate / ref.bachelor - 1) * 100
show(ref.round(0), "Reference: all fields with a readable median (mapped or not), workers-weighted mean of field medians by highest degree")
tc = PAY[PAY.group_1.isin(list(TEST) + CONTRAST)].sort_values(["group_1", "band"])
show(tc[["band"] + pcols].round(1).reset_index(drop=True), "Test and contrast fields, both bands")

**Reference: all fields with a readable median (mapped or not), workers-weighted mean of field medians by highest degree**

degree_level,bachelor,graduate,premium %
age_band,,,
28-34,"99,944.0","108,907.0",9.0
35-64,"102,056.0","134,686.0",32.0


**Test and contrast fields, both bands**

,band,group_1,field,n_b,n_g,median_b,moe_b,median_g,moe_g,premium %,z,verdict,mean premium %,graduate share %
0,28-34,2102,Computer Science,382.0,122.0,"122,891.0","9,883.3","167,579.0","23,896.6",36.4,2.8,readable,46.8,22.7
1,35-64,2102,Computer Science,696.0,366.0,"128,312.0","12,784.2","182,745.0","15,381.9",42.4,4.5,readable,39.4,33.5
2,28-34,3301,English Language And Literature,324.0,261.0,"83,958.0","4,964.3","94,418.0","5,252.5",12.5,2.4,readable,0.2,43.3
3,35-64,3301,English Language And Literature,637.0,829.0,"109,778.0","10,489.3","111,719.0","5,458.8",1.8,0.3,no readable difference,11.7,57.2
4,28-34,3600,Biology,179.0,287.0,"77,561.0","11,190.6","89,375.0","6,252.9",15.2,1.5,no readable difference,26.1,61.5
5,35-64,3600,Biology,195.0,617.0,"89,205.0","8,450.2","156,407.0","7,371.1",75.3,9.9,readable,100.8,75.4
6,28-34,5200,Psychology,376.0,448.0,"83,789.0","7,755.9","91,372.0","8,216.0",9.1,1.1,no readable difference,8.3,54.2
7,35-64,5200,Psychology,660.0,855.0,"83,789.0","4,636.3","117,314.0","5,729.8",40.0,7.5,readable,36.8,56.4
8,28-34,5301,Criminal Justice And Fire Protection,167.0,58.0,"64,435.0","7,747.6","91,651.0","4,111.5",42.2,5.1,readable,43.8,25.2
9,35-64,5301,Criminal Justice And Fire Protection,268.0,113.0,"90,686.0","10,204.5","101,525.0","12,984.9",12.0,1.1,no readable difference,34.1,32.1


## 6. Checks

In [14]:
chk = []
chk.append(["row accounting: fact rows / unsuppressed / with earnings (yml 21,791 / 3,260 / 2,618)", f"{len(G):,} / {int((~G.suppressed).sum()):,} / {int((~G.earnings_suppressed).sum()):,}"])
grp = G[~G.suppressed].groupby(["cut", "group_1", "age_band", "degree_level"]).agg(s=("share_of_group", "sum"), w=("workers", "sum"), gw=("group_workers", "first"))
chk.append(["shares ≤ 1 per group and Σ workers = share × group_workers (max |gap|)", f"max Σshare {grp.s.max():.4f}; max |Σw − Σshare·gw| {(grp.w - grp.s * grp.gw).abs().max():.3f}"])
chk.append(["no workers where suppressed; workers where not", f"{int(G[G.suppressed].workers.notna().sum())} / {int(G[~G.suppressed].workers.isna().sum())}"])
t27 = entry_mix(D[(D.degree_level == "bachelor") & (D.age_band == "22-27")])
under_rows = ["below associate's", "associate's"]
u27_w = t27.loc[under_rows, "weighted, unsuppressed rows %"].sum()
u27_u = t27.loc[under_rows, "unweighted, all rows %"].sum()
chk.append(["underemployment, bachelor's 22-27, below bachelor's entry (step 4 §9: 36.2% not enrolled; card 38.8% with students)",
            f"weighted {u27_w:.1f}%, unweighted {u27_u:.1f}%"])
p = G[(G.cut == "field_x_linked") & (G.group_1 == "5200") & (G.degree_level == "bachelor") & (G.age_band == "22-34")].set_index("group_2")
chk.append(["trace Psychology 22-34: linked workers ÷ group_workers = share (fact)", f"{p.loc['linked', 'workers'] / p.loc['linked', 'group_workers'] * 100:.2f}% vs {p.loc['linked', 'share_of_group'] * 100:.2f}%; n {int(p.loc['linked', 'n_unweighted'])}"])
psy = pairs[pairs.fod1p == "5200"].soc_code.tolist()
chk.append(["trace Psychology linked set = bridge rows under 42.01 / 42.27 / 42.2812 / 42.2813", ", ".join(sorted(psy))])
rn = T[T.field == "6107"].iloc[0]
chk.append(["plausibility: Nursing lift (RN should dominate; expect a pathway)", f"linked {rn['linked share %']:.1f}%, base {rn['base rate %']:.1f}%, lift {rn.lift:.1f}, top {rn['top linked job']}"])
chk.append(["linked (field, job) pairs rebuilt vs the fact's fields with a linked row", f"{pairs.fod1p.nunique()} fields vs {G[(G.cut == 'field_x_linked')].group_1.nunique()}"])
pz = ft[(ft.group_1 == "5200") & (ft.age_band == "35-64")].set_index("degree_level")
z_hand = (pz.loc["graduate", "median_earnings_2024usd"] - pz.loc["bachelor", "median_earnings_2024usd"]) / np.sqrt((pz.loc["graduate", "moe_median_earnings"] / 1.645) ** 2 + (pz.loc["bachelor", "moe_median_earnings"] / 1.645) ** 2)
z_tab = PAY[(PAY.group_1 == "5200") & (PAY.band == "35-64")].z.iloc[0]
chk.append(["trace Psychology 35-64 graduate premium Z by hand vs table (§5c)", f"{z_hand:.2f} vs {z_tab:.2f}; medians ${pz.loc['bachelor', 'median_earnings_2024usd']:,.0f} → ${pz.loc['graduate', 'median_earnings_2024usd']:,.0f}"])
chk.append(["§5c row accounting: 78 mapped fields per band = readable + no difference + unreadable", "; ".join(f"{b}: {len(PAY[PAY.band == b])} = " + " + ".join(str(v) for v in PAY[PAY.band == b].verdict.value_counts().reindex(['readable', 'no readable difference', 'unreadable (a median suppressed)']).fillna(0).astype(int)) for b in ["28-34", "35-64"])])
show(pd.DataFrame(chk, columns=["check", "result"]), "Checks")

**Checks**

,check,result
0,row accounting: fact rows / unsuppressed / with earnings...,"21,791 / 3,260 / 2,618"
1,shares ≤ 1 per group and Σ workers = share × group_worke...,max Σshare 1.0000; max |Σw − Σshare·gw| 0.000
2,no workers where suppressed; workers where not,0 / 0
3,"underemployment, bachelor's 22-27, below bachelor's entr...","weighted 23.4%, unweighted 36.5%"
4,trace Psychology 22-34: linked workers ÷ group_workers =...,4.31% vs 4.31%; n 47
5,trace Psychology linked set = bridge rows under 42.01 / ...,"11-9199, 15-2041, 19-1029, 19-1042, 19-1099, 19-4061, 21..."
6,plausibility: Nursing lift (RN should dominate; expect a...,"linked 73.9%, base 3.2%, lift 23.3, top Registered Nurses"
7,"linked (field, job) pairs rebuilt vs the fact's fields w...",75 fields vs 78
8,trace Psychology 35-64 graduate premium Z by hand vs tab...,"7.48 vs 7.48; medians $83,789 → $117,314"
9,§5c row accounting: 78 mapped fields per band = readable...,28-34: 77 = 9 + 34 + 34; 35-64: 78 = 30 + 33 + 15


## 7. Findings (from the tables above)

1. **Who.** 489,032 employed, not-enrolled NYC residents aged 22–34 hold a bachelor's and nothing higher (203 k aged 22–27, 286 k aged 28–34); 225 k more hold a graduate degree. The 78 mapped fields cover 98.1% of them. Cells under 30 responses are suppressed, which leaves 152 of 541 landed jobs readable (86% of workers) and 27 of 78 fields with a readable linked share.
2. **Where young bachelor's holders work.** Software Developers 5.2%, Managers All Other 4.0%, Registered Nurses 2.7%, Management Analysts 2.4%, Accountants 2.3%, Market Research Analysts 2.2%, Marketing Managers 2.1%, Financial Analysts 2.1%, Customer Service Representatives 1.9%, Financial Managers 1.9%. 60% of readable workers are in degree-reachable jobs. Underemployment (job's BLS entry below a bachelor's) reproduces step 4's 36.2% only on unweighted rows (36.5% at 22–27; 36.6% at 22–34; 40% at 35–64): the weighted figure over readable rows is 23% because suppression removes exactly the small sub-bachelor's jobs. The fact cannot give a weighted, MOE-bearing underemployment number; that needs a `degree_x_entry_education` cut (step 4 follow-up). Associate's holders 22–34 (61 k) are in Police Officers, Retail Salespersons, Teaching Assistants, Security Guards, Home Health Aides; 11% in degree-reachable jobs, 42% in broad Census codes whose entry cannot be judged.
3. **What each field's graduates do.** Professional fields concentrate: Nursing 82% in its top three families (75% Healthcare Practitioners), Computer Science 74% (57% Computer and Mathematical), Finance 69%, Accounting 69% (47% Business and Financial), Economics 65%, Marketing 65%. The test fields spread: Psychology 44% (Management 17%, Office and Administrative Support 15%, Business and Financial 12%, Education 9%, Healthcare 8%, Sales 8%, Arts 7%, Community and Social Service 6%), Political Science 50% (Business and Financial 23%, Management 17%), Criminal Justice 42% (Protective Service 16%, Office 15%, Management 11%; 58% of its workers sit in families under 30 responses). Biology 35%.
4. **The D16 test (§3e).** Of the 27 fields with a readable linked share, 17 are pathways (lift ≥ 3: Nursing 74% linked, lift 23; Architecture 57; Civil Engineering 42; Accounting 47%, lift 6.9; Computer Science 53%, lift 6.0; Finance 41%, 4.7; Marketing 33%, 3.1; English 9% but lift 5.9 through Writers and Authors), 2 weak (Economics 2.3, Mass Media 2.1) and 5 conventions: Psychology lift 1.0, Political Science 1.2, Criminal Justice 1.4, Business Management and Administration 1.4, General Business 1.2. 51 fields (114 k workers, 23% of the population) have fewer than 30 workers in any linked job. Psychology: 4.3% ± 1.9 of its bachelor's-only workers 22–34 hold a linked job against a 4.2% base rate; its Management share is 17.0% against 18.3% for all bachelor's holders (lift 0.9), and the family of its named job (`19-0000`) holds 2.2% against 2.3% (lift 1.0). Its real affinities are Community and Social Service (3.8×), Healthcare Practitioners (2.2×), Education (1.7×), Office (1.6×). Political Science: 4.9% linked vs 4.1% base; Management 17% (0.9×); Business and Financial 23% (1.2×). Criminal Justice: the family rule says B (Protective Service 16%, 13× all bachelor's), but the bridge's protective jobs are the supervisors (`33-1012`: 1 response at 22–34) while the family is Police Officers `33-3050`, a broad Census code with a high-school entry; lift on the bridge's own jobs is 1.4. So neither D16 weight describes the three fields: A puts their supply on a job they hold at the base rate, B would put Psychology's 4,129 awards on a job with 291 openings that its graduates hold at the base rate too. Linked shares are stable with age for the conventions (Psychology 4.3 → 4.5% at 35–64, Political Science 4.9 → 6.0%) and fall for the pathways as workers move into management (Computer Science 57 → 36%, Accounting 51 → 34%, Finance 46 → 28%, Economics 29 → 10%).
5. **Earnings and the graduate route.** Median full-time year-round earnings, bachelor's-only 22–34, 2024 $: Computer Engineering $140.5 k, Computer Science, Finance, Economics and Mathematics ≈ $122 k, Physics $107 k; Criminal Justice $58.7 k, Social Work $58.8 k, Education $60–61 k, Drama $67 k; workers-weighted mean of field medians $90.2 k. Holding a linked job pays in the pathway fields (Computer Engineering +127%, Computer Science +67%, Nursing +53%, Finance +52%) and barely or not in the conventions and the generalist business fields (Accounting +6%, Economics −7%, Criminal Justice −3%; Psychology's +40% rests on 47 responses). At 35–64, 46% of bachelor's-or-higher holders have a graduate degree: Biology 75%, History 66%, Political Science 64%, English 57%, Psychology 56%, Economics 49%, against Accounting 28%, Criminal Justice 32%, Nursing 33%, Computer Science 34%. Half of the Psychology and Political Science fields leave the bachelor's-only population, which is part of why their bachelor's-only destinations look like everyone else's.
6. **Checks.** Rows 21,791 / 3,260 / 2,618 as the yml says; Σ share ≤ 1 and Σ workers = share × group_workers to 0.000 on every readable group; no workers where suppressed; Psychology's linked share recomputed from its rows (4.31%); its linked set rebuilt from the bridge (11-9199, 15-2041, 19-1029, 19-1042, 19-1099, 19-4061, 21-1xxx); Nursing lift 23 with Registered Nurses on top; 75 fields with bridge rows vs 78 in the fact (three mapped fields have no bachelor's bridge row, every worker `not_linked` or `job_not_detailed`); underemployment 36.5% unweighted vs 36.2% (weighted-readable 23%, explained in 2).
7. **Limits.** Field of degree is the bachelor's field, self-reported; the job is a Census code (474 detailed, 67 group codes that cannot be judged: 11–25% of each test field); NYC residents of any college, not NYC graduates; readable cells only (27 of 78 fields for the test); 90% MOEs; the base rate counts suppressed jobs as 0, which flatters the lift slightly for fields with many small linked jobs; the family test cannot see the named job itself (D17 rejected that grain).
8. The graduate-degree premium (§5c, plan §3h): it pays where it is the licence to a profession, and it shows late. At 35–64, 30 of the 78 mapped fields have a readable premium (Census two-estimate test at 90%), every one positive, median +43%; 33 show no readable difference and 15 are unreadable (a median suppressed). Over every field with a readable median, graduate holders' field medians average $134.7 k against $102.1 k for bachelor's-only holders (+32%). Largest: Elementary Education +104% ($51 k → $105 k; New York's professional teaching certificate requires a master's, so the bachelor's-only holders are the ones outside the classroom), Chemical Engineering +98%, Management Information Systems +86%, Biology +75% ($89 k → $156 k, the medical route), Finance +66%, Physics +60%, Electrical Engineering +56%, Mathematics +55%, Economics +50%, International Relations +49%, General Education +46%, Business Management and Administration +45%, Computer Science +42%, Political Science +41% and Psychology +40% ($84 k → $117 k), General Business +39%, History +33%, Accounting +28%, Nursing +22%. No readable difference at 35–64: English +2%, Marketing −7%, the arts and media fields (Fine Arts, Graphic Design, Drama, Journalism, Music, Film), Criminal Justice +12%, Architecture +13%, Civil Engineering +14%, Philosophy +10%. At 28–34 the premium is mostly not yet visible: 9 readable fields (median +36%: General Business +47%, Criminal Justice +42%, Accounting +42%, Computer Science +36%, Mechanical Engineering +27%, Finance +15%, English +13%), 34 with no readable difference (Psychology +9%, Political Science +11%, Nursing +5%, Economics −2%, Biology +15%, Business Management +12%), 34 unreadable; all fields +9%. Reading: in the dispersed fields (Psychology, Political Science, Business Management, General Business; D20) the graduate degree is the step that turns a spread-out bachelor's into a professional track, visible by 35–64 and not before; in the pathway fields (Nursing, Computer Science, Accounting, Finance) the bachelor's already pays and the graduate premium is smaller and later; in the arts, media and design fields it does not show at all. This is what the major page for a dispersed major should say beside its ACS destinations. Checks: Psychology 35–64 Z by hand 7.48 = table; no mean/median sign disagreement on any readable field; 77 fields at 28–34 (Data Science and Data Analytics `4009` has no 28–34 row) and 78 at 35–64 account exactly. Limits: a premium, not a return (selection into graduate study, tuition and years out of work are not counted); the graduate field is unknown (Biology's premium is medicine, Political Science's is law, not a master's in the field); pooled 2020–2024, full-time year-round workers, 2024 dollars; MOE test at 90% with n as low as 30 per cell.
9. **D16 → D20, decided.** The lift test separates the bridge into pathways (professional and technical majors, where graduates-per-opening is a real competition measure) and conventions (Psychology, Political Science, Criminal Justice, the two general business fields), where the S11 ratio counts graduates on jobs they hold at the population rate. The recommendation was neither B nor C but a third `link_status`, and I chose it (logged as D20): `dispersed` for bachelor's majors whose field's lift is under 1.5, built as `core.int_major_field_lift` plus `field_fod1p` / `field_lift` / `field_verdict` on `dim_major_award` (73 majors, 12,067 of the 2024 bachelor's awards; a major under two fields takes the higher readable lift). Their page shows ACS destinations by family and earnings instead of a ratio; nb 42 was re-run without them (Managers All Other 9.9 → 3.1 graduates per opening; "growing · many" 22,306 → 13,758 graduates; the crowded box unchanged).